# ISOM 835 · Midterm Model Competition — Starter Notebook
**Hotel cancellations · launches Mon Oct 26 · leaderboard closes Sun Nov 8, 11:59 PM · Prof. Hasan Arslan**

This notebook gets a leak-free submission on the board in ten minutes. It reproduces the **logistic baseline** benchmark exactly; everything after that is yours.

> **Frame it.** *Unit:* one booking · *Target:* `is_canceled` · *Metric:* ROC-AUC on a hidden half of `test.csv` · *Decision (for the memo):* overbooking and deposit policy.

**Setup:** download `train.csv`, `test.csv`, and `sample_submission.csv` from the competition's Data tab and upload them here (folder icon on the left → upload), or mount Drive.

In [ ]:
# Environment check — run this cell first. If it fails in Colab: run  !pip install -q -U scikit-learn pandas  then Runtime → Restart session.
import sys, re, sklearn, pandas as pd, numpy as np
need = {'scikit-learn': ('1.6', sklearn.__version__), 'pandas': ('2.2', pd.__version__), 'numpy': ('1.26', np.__version__)}
v = lambda s: tuple(int(x) for x in re.findall(r'\d+', s)[:2])
old = {k: have for k, (want, have) in need.items() if v(have) < v(want)}
assert not old, f'please upgrade {old}: !pip install -q -U ' + ' '.join(old)
print(f'Python {sys.version.split()[0]} ·', ' · '.join(f'{k} {have}' for k, (_, have) in need.items()), '✓')

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

train = pd.read_csv('train.csv'); test = pd.read_csv('test.csv')
print(train.shape, test.shape, f'cancel rate {train.is_canceled.mean():.3f}')
X, y = train.drop(columns=['id', 'is_canceled']), train['is_canceled']

## 1. Look before you model
The three leak columns are already gone. Three more deserve a thought: `agent` and `company` are IDs (high-cardinality, many missing) and `country` has ~180 values. One-hot with `min_frequency` keeps them honest.

In [ ]:
print(X.isna().mean().sort_values(ascending=False).head(6).round(3))
print(train.groupby('deposit_type')['is_canceled'].mean().round(3))
print(train.groupby(pd.cut(train['lead_time'], [-1, 7, 30, 90, 180, 800]), observed=True)['is_canceled'].mean().round(3))

## 2. The Session 3 pipeline, unchanged

In [ ]:
num = X.select_dtypes(include='number').columns.tolist()
cat = [c for c in X.columns if c not in num]
prep = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='missing')), ('oh', OneHotEncoder(handle_unknown='ignore', min_frequency=20))]), cat),
])
model = Pipeline([('prep', prep), ('clf', LogisticRegression(max_iter=3000))])
cv = StratifiedKFold(5, shuffle=True, random_state=835)
scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f'5-fold AUC: {scores.mean():.4f} ± {scores.std():.4f}   ← trust this more than the public leaderboard')

## 3. Fit on everything, write the submission

In [ ]:
model.fit(X, y)
sub = pd.DataFrame({'id': test['id'], 'is_canceled': model.predict_proba(test.drop(columns=['id']))[:, 1]})
sub.to_csv('submission.csv', index=False)
print(sub.head(), sub.shape)
# Download submission.csv (folder icon → ⋮ → Download) and submit it on Kaggle. Expect ≈ 0.89 AUC.

## 4. Where the points are
1. **Swap the last step** for `HistGradientBoostingClassifier` (Session 8) with early stopping — the reference lands near 0.96.
2. **Engineer features** (Session 3): total nights, weekend share, `previous_cancellations > 0`, lead-time buckets, room-type mismatch is gone but `booking_changes` and `total_of_special_requests` are not.
3. **Validate like production** (Session 9): if you suspect repeated agents, try `GroupKFold` by `agent` and see whether your CV score holds.
4. **Do not tune on the public leaderboard.** Five submissions a day is plenty if your CV is honest.

Your memo (one page, due Nov 9): what you tried, what worked, what leaked, and one thing you would do with two more weeks.